# 04_quality_review.ipynb — 分類品質レビュー

**役割**: 03_main_run.ipynb の出力を使って、LLM 分類の品質を確認する。

## 観点
1. confidence 低い件のサンプル抽出
2. perspective_match=False (ユーザと修理者で視点が異なる) のサンプル
3. is_misjudged=True (responsibility 食い違い) のサンプル
4. LLM ハルシネーション検出 (体系外コードが出ていないか)
5. insufficient_info=True のレコードレビュー
6. サービスレコードの抽出と確認

出力 CSV を直接読むのではなく、入力 JSON から再構築する。
(CSV からだと dict カラムが文字列化されているため)

## 1. データ読み込み

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from codes_loader import load_codes
from derive_metrics import build_derived_dataframe

YAML_PATH = REPO_ROOT / "config" / "classification_codes.yaml"
PREPARED_DIR = REPO_ROOT / "outputs" / "prepared_data"
DIFY_DIR = REPO_ROOT / "outputs" / "dify_results"

# pandas 表示設定 (列を切らない)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_columns", 50)

TIMESTAMP: str | None = None
if TIMESTAMP is None:
    candidates = sorted(PREPARED_DIR.glob("*_records.json"))
    if not candidates:
        raise FileNotFoundError(f"records.json が {PREPARED_DIR} に見つかりません")
    TIMESTAMP = candidates[-1].name.removesuffix("_records.json")
print(f"TIMESTAMP: {TIMESTAMP}")

records = json.loads((PREPARED_DIR / f"{TIMESTAMP}_records.json").read_text(encoding="utf-8"))
classifications = json.loads((DIFY_DIR / f"{TIMESTAMP}_classifications.json").read_text(encoding="utf-8"))
codebook = load_codes(YAML_PATH)

df = build_derived_dataframe(records, classifications, codebook)
print(f"行数: {len(df)}")

## 2. Confidence 低い件のレビュー

min_confidence < 0.7 のレコードを抽出。LLM の判定が不安定だった可能性が高いので、ユーザの目で見直す。

In [ ]:
LOW_CONF_THRESHOLD = 0.7

low_conf = df[df["min_confidence"] < LOW_CONF_THRESHOLD].sort_values("min_confidence")
print(f"min_confidence < {LOW_CONF_THRESHOLD} : {len(low_conf)} 件")
low_conf[[
    "repair_id", "sub_id", "product_type",
    "user_text", "repair_text",
    "user_failure_code", "repair_failure_code",
    "min_confidence",
    "user_confidence", "repair_confidence",
    "reproduction_confidence",
]].head(20)

## 3. perspective_match=False (視点不一致) のサンプル

ユーザの訴えと修理者の判定でコードが異なる。多くは「ユーザが症状で訴え、修理者が原因部位で記述する」差なので必ずしも問題ではないが、傾向を確認する。

In [ ]:
mismatch = df[~df["perspective_match"].fillna(True)]
print(f"perspective_match=False : {len(mismatch)} 件 ({len(mismatch)/len(df):.1%})")
mismatch[[
    "repair_id", "sub_id", "product_type",
    "user_text",
    "user_failure_code", "user_failure_name",
    "repair_text",
    "repair_failure_code", "repair_failure_name",
]].head(20)

### 視点不一致のコード遷移パターン (ユーザ → 修理者)

どのコードからどのコードへの遷移が多いかを集計。

In [ ]:
transition = (
    mismatch.groupby(["user_failure_code", "repair_failure_code"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(15)
)
transition

## 4. is_misjudged=True (responsibility 食い違い)

ユーザ視点と修理者視点で「メーカー責任」/「ユーザ起因」の判定が異なる。

**注意**: 現状の classification_codes.yaml では responsibility は M012/M013/M014/M015 の4コードのみ付与済み。それ以外は未付与のため False 扱いになっており、is_misjudged はセンサー内/外、ファインダー内/外の混同のみが現れる。yaml 更新後に再評価が必要。

In [ ]:
misjudged = df[df["is_misjudged"].fillna(False)]
print(f"is_misjudged=True : {len(misjudged)} 件")
misjudged[[
    "repair_id", "sub_id", "product_type",
    "user_text", "repair_text",
    "user_failure_code", "is_manufacturer_responsibility_user",
    "repair_failure_code", "is_manufacturer_responsibility_repair",
]].head(20)

## 5. LLM ハルシネーション検出 (体系外コード)

Dify が返したコードが yaml に存在しないものでないか確認する。
コード妥当性は `codebook.is_valid_failure_code` で検証する。

In [ ]:
def _check_valid(code, product_type):
    if not isinstance(code, str) or not isinstance(product_type, str):
        return False
    try:
        return codebook.is_valid_failure_code(code, product_type)
    except ValueError:
        return False

df_check = df.copy()
df_check["user_code_valid"] = df_check.apply(
    lambda r: _check_valid(r["user_failure_code"], r["product_type"]), axis=1
)
df_check["repair_code_valid"] = df_check.apply(
    lambda r: _check_valid(r["repair_failure_code"], r["product_type"]), axis=1
)

invalid_user = df_check[~df_check["user_code_valid"]]
invalid_repair = df_check[~df_check["repair_code_valid"]]
print(f"ユーザ視点コード 体系外: {len(invalid_user)} 件")
print(f"修理者視点コード 体系外: {len(invalid_repair)} 件")

if len(invalid_user) + len(invalid_repair) > 0:
    print("\n--- 体系外コードのサンプル ---")
    invalid_combined = pd.concat([invalid_user, invalid_repair]).drop_duplicates(
        subset=["repair_id", "sub_id"]
    )
    display(invalid_combined[[
        "repair_id", "sub_id", "product_type",
        "user_failure_code", "user_code_valid",
        "repair_failure_code", "repair_code_valid",
        "user_text", "repair_text",
    ]].head(20))

## 6. insufficient_info=True のレコード

LLM が「情報不足で判定できない」とフラグを立てたケース。

In [ ]:
insufficient = df[
    df.get("user_insufficient_info", pd.Series(False, index=df.index)).fillna(False)
    | df.get("repair_insufficient_info", pd.Series(False, index=df.index)).fillna(False)
]
print(f"insufficient_info=True : {len(insufficient)} 件")
insufficient[[
    "repair_id", "sub_id", "product_type",
    "user_text", "repair_text",
    "user_failure_code", "user_insufficient_info",
    "repair_failure_code", "repair_insufficient_info",
]].head(20)

## 7. サービスレコード抽出

「故障ではなく定期的サービス」のレコード。修理者視点が M042-M045 / L032-L035 のもの。

In [ ]:
service = df[df["is_service_record"].fillna(False)]
print(f"サービスレコード: {len(service)} 件 ({len(service)/len(df):.1%})")
service_breakdown = (
    service.groupby(["product_type", "repair_failure_code", "repair_failure_name"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
service_breakdown

## 8. 再現状況の分布

In [ ]:
repro = df["reproduction_status"].value_counts(dropna=False)
print(repro.to_string())
print(f"\n再現率 (reproduced / total): {(df['reproduction_status'] == 'reproduced').mean():.1%}")

## 9. 全体トレンド (修理者視点ベース)

In [ ]:
summary = {
    "総レコード数 (sub_id 単位)": len(df),
    "unique repair_id": df["repair_id"].nunique(),
    "メーカー責任率 (修理者視点)": f"{df['is_manufacturer_responsibility_repair'].fillna(False).mean():.1%}",
    "再現率": f"{(df['reproduction_status'] == 'reproduced').mean():.1%}",
    "環境要因あり率": f"{df['has_harsh_env'].fillna(False).mean():.1%}",
    "修理者環境確認率": f"{df['has_repair_confirmed_env'].fillna(False).mean():.1%}",
    "サービスレコード率": f"{df['is_service_record'].fillna(False).mean():.1%}",
    "視点一致率": f"{df['perspective_match'].fillna(False).mean():.1%}",
    "is_misjudged 率": f"{df['is_misjudged'].fillna(False).mean():.1%}",
}
for k, v in summary.items():
    print(f"  {k:30s}: {v}")

## レビュー完了

ここで気になる傾向 (体系外コードが多い、特定コードへの偏り、低 confidence の集中など) があれば、Dify プロンプトの調整や yaml の見直しに繋げる。